<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); padding: 40px 32px; border-radius: 16px; margin-bottom: 8px;">
  <h1 style="color: #fff; font-size: 2.4em; margin: 0 0 8px 0; font-weight: 700; letter-spacing: -1px;">⚖️ LexAssist</h1>
  <p style="color: #a78bfa; font-size: 1.15em; margin: 0 0 20px 0; font-weight: 500;">Advanced RAG System · Indian Legal & Government Schemes Navigator</p>
  <div style="display: flex; gap: 10px; flex-wrap: wrap;">
    <span style="background:#4c1d95; color:#ddd6fe; padding:4px 12px; border-radius:20px; font-size:0.8em;">🦜 LangChain</span>
    <span style="background:#1e3a5f; color:#bfdbfe; padding:4px 12px; border-radius:20px; font-size:0.8em;">🔷 ChromaDB</span>
    <span style="background:#1a3325; color:#a7f3d0; padding:4px 12px; border-radius:20px; font-size:0.8em;">🤗 BGE Embeddings</span>
    <span style="background:#3b1f1f; color:#fca5a5; padding:4px 12px; border-radius:20px; font-size:0.8em;">⚡ Groq LLM</span>
    <span style="background:#2d2b05; color:#fef08a; padding:4px 12px; border-radius:20px; font-size:0.8em;">📊 RAGAS Eval</span>
  </div>
</div>

---

## 🗺️ What This Notebook Covers

| Stage | What We Build | Complexity |
|-------|--------------|------------|
| **1. Ingestion** | DirectoryLoader → PDF parsing → metadata enrichment | Beginner |
| **2. Chunking** | RecursiveCharacterTextSplitter + chunk analysis | Beginner |
| **3. Embeddings** | BGE-large via HuggingFace Endpoints | Intermediate |
| **4. Vector Store** | Chroma with persistence | Intermediate |
| **5. Basic RAG** | Retriever → Prompt → LLM chain | Intermediate |
| **6. Advanced Retrieval** | HyDE + Multi-query + Reranking | Advanced |
| **7. Conversational RAG** | Memory-aware multi-turn chain | Advanced |
| **8. RAGAS Evaluation** | Faithfulness, Relevancy, Precision, Recall | Advanced |

> 💡 **Run cells in order top-to-bottom.** Each stage builds on the previous one.

---
## 📦 Stage 0 — Installation & Imports

Install all required packages. Run once, then restart the kernel.

In [1]:
# Run once — comment out after first run
# !pip install langchain langchain-community langchain-core langchain-groq \
#              langchain-huggingface langchain-chroma chromadb pypdf \
#              sentence-transformers python-dotenv streamlit ragas datasets \
#              rank-bm25 --quiet

In [2]:
# ── Core
import os, time, warnings
from pathlib import Path
from dotenv import load_dotenv

# ── LangChain — Document loading & splitting
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── LangChain — Embeddings & Vector Store
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma

# ── LangChain — LLM
from langchain_groq import ChatGroq

# ── LangChain — Prompt & Chain
from langchain_core.prompts import (
    ChatPromptTemplate, SystemMessagePromptTemplate,
    HumanMessagePromptTemplate, PromptTemplate, MessagesPlaceholder
)
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# ── Advanced Retrieval
from langchain.retrievers import ContextualCompressionRetriever, MultiQueryRetriever
from langchain.retrievers.document_compressors.chain_extract import LLMChainExtractor

# ── Memory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

warnings.filterwarnings("ignore")
load_dotenv()

print("✅ All imports successful")

✅ All imports successful


---
## ⚙️ Stage 0.1 — Configuration

All tuneable parameters in one place. Change values here, not scattered across cells.

In [3]:
# ─────────────────────────────────────────────
#  PROJECT CONFIGURATION  — edit here only
# ─────────────────────────────────────────────

DATA_DIR          = "Data"                        # folder with your PDFs
CHROMA_DIR        = "./chroma_db"                 # persisted vector store
COLLECTION_NAME   = "lexassist"

# Chunking
CHUNK_SIZE        = 1000
CHUNK_OVERLAP     = 200

# Embedding model (free, runs via HF Inference API)
EMBED_MODEL       = "BAAI/bge-large-en-v1.5"      # 1024-dim, strong retrieval

# LLM  (get free key → console.groq.com)
GROQ_MODEL        = "openai/gpt-oss-120b"              # fast & free

# Retrieval
RETRIEVER_K       = 4                             # chunks fetched per query
MULTIQUERY_K      = 3                             # sub-queries for multi-query

# ─────────────────────────────────────────────
print(f"📁 Data dir   : {DATA_DIR}")
print(f"🗃️  Chroma dir : {CHROMA_DIR}")
print(f"🤗 Embed model: {EMBED_MODEL}")
print(f"⚡ LLM model  : {GROQ_MODEL}")

📁 Data dir   : Data
🗃️  Chroma dir : ./chroma_db
🤗 Embed model: BAAI/bge-large-en-v1.5
⚡ LLM model  : openai/gpt-oss-120b


---
## 📄 Stage 1 — Document Loading

**What happens here:**  
`DirectoryLoader` walks your `Data/` folder, hands every `.pdf` to `PyPDFLoader`,  
and returns a list of `Document` objects — each containing raw text and metadata  
(filename, page number, author, creation date).

```
Data/
 ├── ipc_bare_act.pdf         →  [Document(page=0), Document(page=1), ...]
 ├── constitution_india.pdf   →  [Document(page=0), ...]
 └── pm_kisan_scheme.pdf      →  [Document(page=0), ...]
```

In [4]:
def load_documents(data_dir: str) -> list:
    """Load all PDFs from a directory and return Document list."""
    if not Path(data_dir).exists():
        raise FileNotFoundError(f"Data directory '{data_dir}' not found. Create it and add PDFs.")

    loader = DirectoryLoader(
        data_dir,
        glob="*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True
    )
    docs = loader.load()

    # ── Enrich metadata: add a cleaner source label
    for doc in docs:
        doc.metadata["file_name"] = Path(doc.metadata["source"]).name

    return docs


documents = load_documents(DATA_DIR)

# ── Summary
sources = set(d.metadata["file_name"] for d in documents)
print(f"\n📚 Loaded {len(documents)} pages from {len(sources)} PDF(s)")
print("\nFiles found:")
for s in sources:
    count = sum(1 for d in documents if d.metadata["file_name"] == s)
    print(f"  • {s}  ({count} pages)")

print(f"\n--- First 400 chars of page 1 ---")
print(documents[0].page_content[:400])
print(f"\n--- Metadata ---")
print({k: v for k, v in documents[0].metadata.items() if k != "source"})

100%|██████████| 4/4 [00:11<00:00,  2.87s/it]


📚 Loaded 562 pages from 4 PDF(s)

Files found:
  • constitution.pdf  (404 pages)
  • rti.pdf  (78 pages)
  • consumer.pdf  (22 pages)
  • ipc.pdf  (58 pages)

--- First 400 chars of page 1 ---
  
 
 
 
 THE CONSTITUTION OF INDIA 
[As on       May, 2022] 
2022 
 

--- Metadata ---
{'page': 0, 'file_name': 'constitution.pdf'}


---
## ✂️ Stage 2 — Chunking

### Why chunk at all?
- Embedding models have a **token limit** (~512 tokens). Long pages get truncated.
- Smaller, focused chunks give **more precise retrieval** — you get the exact passage, not a whole page.
- `chunk_overlap=200` means adjacent chunks share 200 characters, preventing answers from being cut at boundaries.

### How `RecursiveCharacterTextSplitter` works
It tries to split on `\n\n` first (paragraphs), then `\n` (lines), then `.` (sentences), then ` ` (words).  
This preserves semantic coherence better than a naive character-count split.

In [5]:
def chunk_documents(docs: list, chunk_size: int, chunk_overlap: int) -> list:
    """Split documents into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],  # priority order
        length_function=len,
        add_start_index=True   # adds char offset to metadata
    )
    return splitter.split_documents(docs)


chunks = chunk_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

# ── Chunk Analysis
lengths = [len(c.page_content) for c in chunks]
print(f"📊 Chunking Report")
print(f"   Total chunks  : {len(chunks)}")
print(f"   Avg length    : {sum(lengths)//len(lengths)} chars")
print(f"   Min / Max     : {min(lengths)} / {max(lengths)} chars")

# ── Overlap verification — chunk n+1 should start with text from chunk n's end
print(f"\n🔍 Overlap check (end of chunk 4 vs start of chunk 5):")
print(f"  Chunk[4] ends with : ...{chunks[4].page_content[-80:].strip()!r}")
print(f"  Chunk[5] starts    : {chunks[5].page_content[:80].strip()!r}...")

📊 Chunking Report
   Total chunks  : 1712
   Avg length    : 824 chars
   Min / Max     : 52 / 999 chars

🔍 Overlap check (end of chunk 4 vs start of chunk 5):
  Chunk[4] ends with : ...'ndia from Pakistan. \n  7. Rights of citizenship of certain migrants to Pakistan.'
  Chunk[5] starts    : '6. Rights of citizenship of certain persons who have migrated to \nIndia from Pak'...


---
## 🔢 Stage 3 — Embeddings

`BAAI/bge-large-en-v1.5` produces **1024-dimensional dense vectors** per chunk.  
These vectors capture semantic meaning — "grant scheme" and "financial assistance" will be close in this space even though they share no words.

> 💡 **Why BGE over OpenAI embeddings?**  
> BGE-large consistently ranks top-3 on the MTEB retrieval leaderboard, it's free via HuggingFace Inference API, and it's specifically optimized for passage retrieval — exactly what RAG needs.

In [6]:
embedding_model = HuggingFaceEndpointEmbeddings(
    model=EMBED_MODEL,
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

# ── Sanity check
test_vec = embedding_model.embed_query("What is a financial grant?")
print(f"✅ Embedding model loaded")
print(f"   Model    : {EMBED_MODEL}")
print(f"   Dimension: {len(test_vec)}")
print(f"   Sample   : [{test_vec[0]:.4f}, {test_vec[1]:.4f}, {test_vec[2]:.4f} ...]")

AttributeError: 'InferenceClient' object has no attribute 'post'

---
## 🗄️ Stage 4 — Vector Store (ChromaDB)

**First run:** embeds all chunks and persists to `./chroma_db/`  
**Subsequent runs:** loads from disk — no re-embedding needed (saves API calls + time)

Under the hood, Chroma stores:
- The raw text of each chunk
- Its 1024-dim embedding vector  
- All metadata (source file, page number, char offset)

In [ ]:
def build_or_load_vectorstore(chunks, embedding_model, persist_dir, collection_name):
    """Build vectorstore from chunks, or reload from disk if it already exists."""
    if Path(persist_dir).exists() and any(Path(persist_dir).iterdir()):
        print(f"♻️  Loading existing vectorstore from {persist_dir}")
        return Chroma(
            persist_directory=persist_dir,
            embedding_function=embedding_model,
            collection_name=collection_name
        )

    print(f"⚙️  Building vectorstore — embedding {len(chunks)} chunks...")
    t0 = time.time()
    vs = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_dir,
        collection_name=collection_name
    )
    print(f"✅ Done in {time.time()-t0:.1f}s  |  {len(chunks)} chunks stored")
    return vs


vectorstore = build_or_load_vectorstore(chunks, embedding_model, CHROMA_DIR, COLLECTION_NAME)

# ── Quick retrieval test
test_results = vectorstore.similarity_search("eligibility criteria for grant", k=2)
print(f"\n🔍 Test retrieval — top 2 results:")
for i, doc in enumerate(test_results):
    print(f"\n  [{i+1}] Source: {doc.metadata.get('file_name','?')}  |  Page: {doc.metadata.get('page','?')}")
    print(f"       {doc.page_content[:200].strip()}...")

♻️  Loading existing vectorstore from ./chroma_db

🔍 Test retrieval — top 2 results:


---
## ⛓️ Stage 5 — Basic RAG Chain

The core loop:

```
User question
     │
     ▼
  Retriever  ──→  top-k chunks  ──→  format_docs()
     │                                     │
     └──────── {question} ─────────────────┤
                                           ▼
                                   Prompt Template
                                   [system + context + question]
                                           │
                                           ▼
                                         LLM
                                           │
                                           ▼
                                    String answer
```

In [ ]:
# ── LLM
llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.1,          # low temp = more factual, less creative
    max_tokens=1024,
    api_key=os.getenv("GROQ_API_KEY")
)

print(f"✅ LLM ready: {GROQ_MODEL}")

✅ LLM ready: openai/gpt-oss-120b


In [ ]:
# ── Prompt Template
SYSTEM_PROMPT = """You are LexAssist, an expert Indian legal and government scheme assistant.
Answer questions STRICTLY from the provided context.
If the context does not contain enough information, reply: "I don't have enough information in the provided documents to answer this."
Always mention which document/section your answer is based on.
Be concise, accurate, and use plain language."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

# ── Retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                     # Max Marginal Relevance — diversity + relevance
    search_kwargs={"k": RETRIEVER_K, "fetch_k": 10}
)

def format_docs(docs: list) -> str:
    """Format retrieved docs into a single context string with source labels."""
    parts = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("file_name", "unknown")
        page   = doc.metadata.get("page", "?")
        parts.append(f"[Source {i}: {source}, page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

# ── Basic RAG Chain
basic_rag_chain = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Basic RAG chain assembled")

✅ Basic RAG chain assembled


In [ ]:
def ask(question: str, chain=basic_rag_chain, show_sources: bool = True):
    """Ask a question and print the answer with source documents."""
    print(f"❓ Question: {question}")
    print("─" * 60)

    answer = chain.invoke(question)
    print(f"🤖 Answer:\n{answer}")

    if show_sources:
        docs = retriever.invoke(question)
        print(f"\n📎 Sources used:")
        for doc in docs:
            print(f"  • {doc.metadata.get('file_name','?')}  (page {doc.metadata.get('page','?')})")
    print()


# ── Test it
ask("What is the scheme for assistance to voluntary organisations?")
ask("Who is eligible to apply for the grant?")

❓ Question: What is the scheme for assistance to voluntary organisations?
────────────────────────────────────────────────────────────
🤖 Answer:
I don't have enough information in the provided documents to answer this.

📎 Sources used:

❓ Question: Who is eligible to apply for the grant?
────────────────────────────────────────────────────────────
🤖 Answer:
I don't have enough information in the provided documents to answer this.

📎 Sources used:



---
## 🚀 Stage 6 — Advanced Retrieval Techniques

Basic retrieval often fails when:
- The user's phrasing differs from document language
- The answer spans multiple disconnected sections  
- Top retrieved chunks are similar to each other (no diversity)

We fix this with three techniques:

| Technique | Problem it solves |
|-----------|------------------|
| **Multi-Query** | User's phrasing doesn't match docs — generate 3 variations |
| **HyDE** | Abstract questions — generate hypothetical answer first, then retrieve |
| **Contextual Compression** | Retrieved chunks have noise — extract only relevant sentences |

In [ ]:
# ──────────────────────────────────────────────
#  TECHNIQUE 1 — Multi-Query Retrieval
#  Generates N paraphrases of the query,
#  retrieves for each, deduplicates results.
# ──────────────────────────────────────────────

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_K}),
    llm=llm,
    include_original=True    # also retrieve for the original query
)

# Test — see what sub-queries get generated
import logging
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

print("🔁 Multi-Query Retrieval test:")
print("  Query: 'financial help for legal organisations'")
docs_mq = multi_query_retriever.invoke("financial help for legal organisations")
print(f"  Retrieved {len(docs_mq)} unique chunks (after dedup)")

🔁 Multi-Query Retrieval test:
  Query: 'financial help for legal organisations'
  Retrieved 0 unique chunks (after dedup)


In [ ]:
# ──────────────────────────────────────────────
#  TECHNIQUE 2 — HyDE  (Hypothetical Document Embeddings)
#  Instead of embedding the question directly,
#  generate a hypothetical answer and embed THAT.
#  The hypothetical answer lives in document space → better retrieval.
# ──────────────────────────────────────────────

hyde_prompt = ChatPromptTemplate.from_template(
    """Generate a SHORT hypothetical passage (2-3 sentences) that would directly answer this question.
Write it as if it were extracted from an official Indian legal document.
Do NOT answer the question yourself — only generate the hypothetical passage.

Question: {question}
Hypothetical passage:"""
)

def hyde_retrieve(question: str, k: int = RETRIEVER_K) -> list:
    """HyDE: generate hypothetical doc → embed it → retrieve real docs."""
    # Step 1: Generate hypothetical answer
    hyp_chain = hyde_prompt | llm | StrOutputParser()
    hypothetical = hyp_chain.invoke({"question": question})

    # Step 2: Retrieve using the hypothetical as query
    docs = vectorstore.similarity_search(hypothetical, k=k)

    return docs, hypothetical


question = "Can a private university apply for the grant?"
docs_hyde, hyp = hyde_retrieve(question)

print(f"🔮 HyDE test")
print(f"  Original query    : {question}")
print(f"  Hypothetical doc  : {hyp[:200]}")
print(f"  Retrieved {len(docs_hyde)} chunks via hypothetical embedding")

🔮 HyDE test
  Original query    : Can a private university apply for the grant?
  Hypothetical doc  : Pursuant to Section 12(3) of the Grant Allocation Act, 2023, a private university (hereinafter “the Applicant”) is expressly permitted to submit an application for the grant, provided it satisfies the
  Retrieved 0 chunks via hypothetical embedding


In [ ]:
# ──────────────────────────────────────────────
#  TECHNIQUE 3 — Contextual Compression
#  After retrieval, strip irrelevant sentences
#  from each chunk using the LLM itself.
#  Result: cleaner context, fewer tokens, less hallucination.
# ──────────────────────────────────────────────

compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": RETRIEVER_K})
)

# ── Build advanced chain using compression retriever
advanced_rag_chain = (
    {"context": compression_retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Advanced RAG chain with contextual compression ready")

# ── Compare basic vs advanced on the same question
q = "What are the conditions for using the grant amount?"
print(f"\n📊 Comparison on: '{q}'")
print("\n[BASIC]")
print(basic_rag_chain.invoke(q))
print("\n[ADVANCED — with compression]")
print(advanced_rag_chain.invoke(q))

✅ Advanced RAG chain with contextual compression ready

📊 Comparison on: 'What are the conditions for using the grant amount?'

[BASIC]
I don't have enough information in the provided documents to answer this.

[ADVANCED — with compression]
I don't have enough information in the provided documents to answer this.


---
## 💬 Stage 7 — Conversational RAG with Memory

Without memory, every question is independent. The user can't say "tell me more about that" or "what about eligibility?" after a previous answer.  

We add two things:
1. **`chat_history`** slot in the prompt — past turns go here
2. **`RunnableWithMessageHistory`** — automatically manages per-session history

In [ ]:
# ── Conversational prompt (adds chat_history slot)
conv_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

# ── Session store — maps session_id → message history
session_store: dict[str, ChatMessageHistory] = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# ── Chain (without history wrapper)
conv_chain_base = (
    RunnableParallel({
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"],
        "chat_history": lambda x: x.get("chat_history", [])
    })
    | conv_prompt
    | llm
    | StrOutputParser()
)

# ── Wrap with history
conv_rag = RunnableWithMessageHistory(
    conv_chain_base,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

print("✅ Conversational RAG with memory ready")

✅ Conversational RAG with memory ready


In [ ]:
# ── Multi-turn demo
SESSION = "demo_session_01"

def chat(question: str, session_id: str = SESSION):
    print(f"👤 User  : {question}")
    resp = conv_rag.invoke(
        {"question": question},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"🤖 LexAssist: {resp}")
    print()


chat("What is the grant scheme about?")
chat("Who can apply for it?")           # 'it' refers to grant scheme — memory needed
chat("What documents do they need?")    # 'they' refers to applicants from prev turn

👤 User  : What is the grant scheme about?
🤖 LexAssist: I don't have enough information in the provided documents to answer this.

👤 User  : Who can apply for it?
🤖 LexAssist: I don't have enough information in the provided documents to answer this.

👤 User  : What documents do they need?
🤖 LexAssist: I don't have enough information in the provided documents to answer this.



---
## 📊 Stage 8 — RAGAS Evaluation

**RAGAS** (RAG Assessment) measures 4 metrics automatically — no human labels needed:

| Metric | What it measures | Ideal |
|--------|-----------------|-------|
| **Faithfulness** | Does answer only use facts from context? | 1.0 |
| **Answer Relevancy** | Is the answer on-topic with the question? | 1.0 |
| **Context Precision** | Was relevant context ranked first? | 1.0 |
| **Context Recall** | Was all important context retrieved? | 1.0 |

> ⚠️ RAGAS requires an OpenAI key by default (it uses GPT for evaluation).  
> The cell below shows the pattern — plug in your key and sample Q&A pairs to run it.

In [ ]:
# ── RAGAS Evaluation
# Requires: pip install ragas datasets
# Requires: OPENAI_API_KEY in .env

def build_ragas_dataset(questions: list[str]) -> dict:
    """Build evaluation dataset by running the RAG chain on each question."""
    dataset = {"question": [], "answer": [], "contexts": [], "ground_truth": []}

    for q in questions:
        retrieved = retriever.invoke(q)
        answer    = basic_rag_chain.invoke(q)

        dataset["question"].append(q)
        dataset["answer"].append(answer)
        dataset["contexts"].append([d.page_content for d in retrieved])
        dataset["ground_truth"].append("")   # fill manually for full eval

    return dataset


# ── Sample evaluation questions (about your documents)
EVAL_QUESTIONS = [
    "What is the purpose of the grant scheme?",
    "Who is eligible to receive the grant?",
    "What conditions must be met before drawing the grant?",
]

print("📋 RAGAS evaluation dataset structure:")
sample = build_ragas_dataset(EVAL_QUESTIONS[:1])
print(f"  Question : {sample['question'][0]}")
print(f"  Answer   : {sample['answer'][0][:200]}...")
print(f"  Contexts : {len(sample['contexts'][0])} chunks retrieved")

print("""
─────────────────────────────────────────────
To run full RAGAS evaluation, uncomment below:
─────────────────────────────────────────────
""")

# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# from datasets import Dataset
#
# full_data = build_ragas_dataset(EVAL_QUESTIONS)
# ragas_dataset = Dataset.from_dict(full_data)
#
# result = evaluate(
#     dataset=ragas_dataset,
#     metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
# )
# print(result)

📋 RAGAS evaluation dataset structure:
  Question : What is the purpose of the grant scheme?
  Answer   : I don't have enough information in the provided documents to answer this....
  Contexts : 0 chunks retrieved

─────────────────────────────────────────────
To run full RAGAS evaluation, uncomment below:
─────────────────────────────────────────────



---
## 🛠️ Stage 9 — Utility Functions

Helper functions you'll reuse in the Streamlit app (`app.py`).

In [ ]:
def get_answer_with_sources(question: str, use_advanced: bool = False) -> dict:
    """
    Single function that returns answer + sources + metadata.
    Used by app.py Streamlit interface.
    """
    chain   = advanced_rag_chain if use_advanced else basic_rag_chain
    t0      = time.time()
    answer  = chain.invoke(question)
    latency = round(time.time() - t0, 2)
    sources = retriever.invoke(question)

    return {
        "answer" : answer,
        "latency": latency,
        "sources": [
            {
                "file" : doc.metadata.get("file_name", "unknown"),
                "page" : doc.metadata.get("page", "?"),
                "text" : doc.page_content[:300]
            }
            for doc in sources
        ]
    }


# ── Final demo
result = get_answer_with_sources("What happens if the grant conditions are violated?")
print(f"Answer  : {result['answer']}")
print(f"Latency : {result['latency']}s")
print(f"Sources : {len(result['sources'])} chunks")

Answer  : I don't have enough information in the provided documents to answer this.
Latency : 0.98s
Sources : 0 chunks


---
## ✅ What You've Built

```
PDF files  →  DirectoryLoader  →  PyPDFLoader  →  Document objects
                                                        │
                                         RecursiveCharacterTextSplitter
                                                        │
                                              21 overlapping chunks
                                                        │
                                       BAAI/bge-large-en-v1.5 embeddings
                                                        │
                                           ChromaDB (persisted to disk)
                                                        │
                              ┌─────────────────────────┤
                              │                         │
                         Basic RAG              Advanced RAG
                         MMR retrieval       Multi-Query + HyDE
                              │              Contextual Compression
                              └────────┬────────────────┘
                                       │
                            Conversational RAG + Memory
                                       │
                               RAGAS Evaluation
                                       │
                               Streamlit UI (app.py)
```

### 🔭 What's next (after this module)
- **Agents** — LLM decides whether to retrieve, calculate, or search the web
- **Tool calling** — RAG + Calculator + Web search + Code executor
- **LangGraph** — Stateful multi-step workflows with conditional branching

---
*LexAssist · Built with LangChain, ChromaDB, Groq, BGE embeddings*